# Phase 2 — Baseline RT-DETR Training (Colab T4)

**Pre-requisite:** Phase 1 complete — dataset in Drive, VIZ 1.A/B/C passed.

**Target metrics:**
- mAP@0.5 ≥ 0.70 on clean test set
- mAP@0.5:0.95 ≥ 0.50
- All class precision ≥ 0.65, recall ≥ 0.60

**If mAP@0.5 < 0.60 after training, do NOT proceed to Phase 3.**

In [ ]:
import os, shutil, torch
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_DIR = '/content/drive/MyDrive/robot-perception'
RESULTS_DIR = f'{PROJECT_DIR}/results/figures'
os.makedirs(RESULTS_DIR, exist_ok=True)

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Step 1 — Copy dataset to local disk

Drive I/O is slow during training. Copy to `/content/` for fast reads.

In [ ]:
LOCAL_DATA = '/content/robot_data'
if os.path.exists(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)
shutil.copytree(f'{PROJECT_DIR}/data/annotated', LOCAL_DATA)

for split in ['train', 'val', 'test']:
    n = len([f for f in os.listdir(f'{LOCAL_DATA}/images/{split}')
             if f.endswith(('.jpg','.png','.jpeg'))])
    print(f'{split}: {n} images')

## Step 2 — Write dataset YAML pointing to local data

In [ ]:
yaml_content = f"""path: {LOCAL_DATA}
train: images/train
val: images/val
test: images/test

nc: 5
names: ['arm', 'leg', 'torso', 'head', 'sensor']
"""

YAML_PATH = '/content/robot_parts.yaml'
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)
print(yaml_content)

## Step 3 — Train RT-DETR-L

In [ ]:
from ultralytics import RTDETR

model = RTDETR('rtdetr-l.pt')  # downloads COCO pretrained weights (~115MB)

results = model.train(
    data=YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=8,          # T4 has 15GB VRAM — increase to 16 if no OOM
    lr0=1e-4,
    weight_decay=1e-4,
    warmup_epochs=3,
    patience=20,      # early stopping
    device=0,         # GPU
    project='/content/runs',
    name='baseline',
    exist_ok=True,
    save=True,
)

## Step 4 — Save best weights to Drive

In [ ]:
import shutil
best_local = '/content/runs/baseline/weights/best.pt'
best_drive = f'{PROJECT_DIR}/models/baseline/best.pt'
os.makedirs(os.path.dirname(best_drive), exist_ok=True)
shutil.copy2(best_local, best_drive)
print(f'Saved: {best_drive}')

## VIZ 2.A — Training curve

In [ ]:
results_csv = '/content/runs/baseline/results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df['epoch'], df['train/box_loss'], label='train box loss')
axes[0].plot(df['epoch'], df['val/box_loss'], label='val box loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('VIZ 2.A — Box loss over training')
axes[0].legend()

axes[1].plot(df['epoch'], df['metrics/mAP50(B)'], label='val mAP@0.5', color='green')
axes[1].axhline(y=0.70, color='red', linestyle='--', label='target (0.70)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mAP@0.5')
axes[1].set_title('Validation mAP over training')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz2a_training_curve.png', dpi=150)
plt.show()
print('⚠️  If val mAP still rising at end → train more epochs.')
print('   Val loss rising but train loss falling → overfitting.')

## Step 5 — Evaluate on clean test set

In [ ]:
# Load best weights
model = RTDETR(f'{PROJECT_DIR}/models/baseline/best.pt')

metrics = model.val(data=YAML_PATH, split='test')

map50     = metrics.box.map50
map5095   = metrics.box.map
precision = metrics.box.p   # per class
recall    = metrics.box.r   # per class

CLASS_NAMES = ['arm', 'leg', 'torso', 'head', 'sensor']

print('=== Clean Test Set Results ===')
print(f'mAP@0.5:       {map50:.4f}  (target ≥ 0.70)')
print(f'mAP@0.5:0.95:  {map5095:.4f}  (target ≥ 0.50)')
print()
print('Per-class precision (target ≥ 0.65):')
for name, p in zip(CLASS_NAMES, precision):
    flag = '✅' if p >= 0.65 else '⚠️ '
    print(f'  {flag} {name}: {p:.4f}')
print()
print('Per-class recall (target ≥ 0.60):')
for name, r in zip(CLASS_NAMES, recall):
    flag = '✅' if r >= 0.60 else '⚠️ '
    print(f'  {flag} {name}: {r:.4f}')

if map50 < 0.60:
    print()
    print('❌ mAP@0.5 < 0.60 — DO NOT proceed to Phase 3.')
    print('   Diagnose: check annotation quality, class balance, try 50 more epochs.')
elif map50 >= 0.70:
    print()
    print('✅ Phase 2 target met. Proceed to Phase 3.')

## VIZ 2.B — Confusion matrix

In [ ]:
# Ultralytics generates a confusion matrix automatically — find and copy it
import glob

cm_path = glob.glob('/content/runs/baseline/confusion_matrix*.png')
if cm_path:
    shutil.copy2(cm_path[0], f'{RESULTS_DIR}/viz2b_confusion_matrix.png')
    from IPython.display import Image
    display(Image(cm_path[0]))
    print('VIZ 2.B saved.')
    print('⚠️  Diagonal should dominate. Off-diagonal = class confusion.')
else:
    print('Confusion matrix not found. Check /content/runs/baseline/ for .png files.')

## VIZ 2.C — 10 qualitative detection examples

In [ ]:
import random
import numpy as np

test_imgs = [f'{LOCAL_DATA}/images/test/{f}'
             for f in os.listdir(f'{LOCAL_DATA}/images/test')
             if f.endswith(('.jpg','.jpeg','.png'))]
sample = random.sample(test_imgs, min(10, len(test_imgs)))

fig, axes = plt.subplots(2, 5, figsize=(25, 10))

for ax, img_path in zip(axes.flatten(), sample):
    result = model(img_path, verbose=False)[0]
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB).copy()
    h, w = img.shape[:2]

    # Draw predictions in blue
    for box, cls, conf in zip(result.boxes.xyxy.cpu().numpy(),
                               result.boxes.cls.cpu().numpy(),
                               result.boxes.conf.cpu().numpy()):
        x1,y1,x2,y2 = map(int, box)
        cv2.rectangle(img,(x1,y1),(x2,y2),(80,80,255),2)
        cv2.putText(img,f'{CLASS_NAMES[int(cls)]} {conf:.2f}',
                    (x1,max(y1-6,0)),cv2.FONT_HERSHEY_SIMPLEX,0.5,(80,80,255),2)

    # Draw ground truth in green
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = f'{LOCAL_DATA}/labels/test/{stem}.txt'
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls = int(parts[0])
                cx,cy,bw,bh = float(parts[1]),float(parts[2]),float(parts[3]),float(parts[4])
                x1=int((cx-bw/2)*w); y1=int((cy-bh/2)*h)
                x2=int((cx+bw/2)*w); y2=int((cy+bh/2)*h)
                cv2.rectangle(img,(x1,y1),(x2,y2),(80,200,80),2)

    ax.imshow(img); ax.axis('off')
    ax.set_title(os.path.basename(img_path)[:20], fontsize=8)

plt.suptitle('VIZ 2.C — GREEN: ground truth | BLUE: predictions', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz2c_qualitative_detections.png', dpi=120)
plt.show()
print('Label each image: GOOD / FALSE_POSITIVE / MISS / WRONG_CLASS')

## Phase 2 Completion Checklist

- [ ] mAP@0.5 ≥ 0.70
- [ ] mAP@0.5:0.95 ≥ 0.50
- [ ] All class precision ≥ 0.65
- [ ] All class recall ≥ 0.60
- [ ] VIZ 2.A saved — training curve
- [ ] VIZ 2.B saved — confusion matrix
- [ ] VIZ 2.C saved — 10 qualitative detections
- [ ] best.pt saved to Drive at `models/baseline/best.pt`

If all checked → open `03_corruption_benchmark.ipynb`